# PII Detection & UC Tagging (v2)

Scans Unity Catalog tables for PII using Microsoft Presidio and applies `pii` tags.

**Features:**
- Works on **serverless compute** (no spaCy dependency)
- All US + APAC entity types (30+ recognizers)
- Custom regex PatternRecognizers (Employee ID, Project Code)
- Optional remote LLM detection for PERSON/LOCATION via Databricks Foundation Model API
- Structured engine analysis via `presidio_structured`
- Column-name-based context-aware score enhancement

**Scope levels:**
- `catalog.schema.table` — single table
- `catalog.schema` — all tables in a schema
- `catalog` — all tables in a catalog

In [0]:
%pip install presidio_analyzer==2.2.362 presidio_structured -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.text("scope", "", "Scope (catalog[.schema[.table]]):")
dbutils.widgets.text("sample_size", "500", "Sample size per table:")
dbutils.widgets.text("score_threshold", "0.5", "Score threshold:")
dbutils.widgets.text("results_table", "", "Results table (optional):")
dbutils.widgets.text("llm_endpoint", "", "LLM endpoint (optional, for PERSON/LOCATION):")
dbutils.widgets.dropdown("llm_debug", "off", ["off", "on"], "LLM Debug Output:")

In [0]:
scope = dbutils.widgets.get("scope").strip()
sample_size = int(dbutils.widgets.get("sample_size"))
score_threshold = float(dbutils.widgets.get("score_threshold"))
results_table_override = dbutils.widgets.get("results_table").strip()
llm_endpoint = dbutils.widgets.get("llm_endpoint").strip()
llm_debug = dbutils.widgets.get("llm_debug").strip().lower() == "on"

assert scope, "scope parameter is required"
print(f"Scope: {scope}")
print(f"Sample size: {sample_size}, Score threshold: {score_threshold}")
print(f"LLM endpoint: {llm_endpoint or '(none -- PERSON/LOCATION detection via LLM disabled)'}")
print(f"LLM debug: {'ON' if llm_debug else 'OFF'}")

Scope: shao_sandbox1.dbdemos_ai_agent
Sample size: 500, Score threshold: 0.9
LLM endpoint: databricks-gpt-5-4-nano
LLM debug: OFF


## Configuration

### Entity-to-tag mapping
Maps Presidio entity types to your UC tag policy's allowed values.
Set value to `None` to detect but skip tagging.

In [0]:
ENTITY_TO_TAG = {
    # -- Generic --
    "PERSON": "name",
    "EMAIL_ADDRESS": "email",
    "CREDIT_CARD": "cc",
    "IBAN_CODE": "iban",
    "IP_ADDRESS": "ipv4",
    "PHONE_NUMBER": "phone",
    "LOCATION": "address",
    "CRYPTO": None,
    # -- US --
    "US_SSN": "ssn",
    "US_PASSPORT": None,
    "US_DRIVER_LICENSE": None,
    "US_BANK_NUMBER": None,
    "US_ITIN": "tax_id",
    "US_MBI": None,
    "US_NPI": None,
    "ABA_ROUTING_NUMBER": None,
    # -- Singapore --
    "SG_NRIC_FIN": "ssn",
    "SG_UEN": None,
    # -- Custom (examples) --
    "EMPLOYEE_ID": None,
    "PROJECT_CODE": None,
}

ALLOWED_ENTITY_TYPES = list(ENTITY_TO_TAG.keys())

## MinimalNlpEngine (no spaCy needed)

A lightweight NLP engine stub that satisfies the `AnalyzerEngine` interface
without requiring spaCy or any NLP model.

In [0]:
from typing import List, Iterable, Iterator, Tuple, Optional
from presidio_analyzer.nlp_engine import NlpEngine, NlpArtifacts


class MinimalNlpEngine(NlpEngine):
    """No-op NLP engine — enables pattern-based recognizers without spaCy."""

    engine_name = "minimal"
    is_available = True

    def __init__(self):
        super().__init__()

    def load(self) -> None:
        pass

    def is_loaded(self) -> bool:
        return True

    def process_text(self, text: str, language: str) -> NlpArtifacts:
        return NlpArtifacts(
            entities=[],
            tokens=[],
            tokens_indices=[],
            lemmas=[],
            nlp_engine=self,
            language=language,
            scores=[],
        )

    def process_batch(
        self, texts: Iterable[str], language: str, **kwargs
    ) -> Iterator[Tuple[str, NlpArtifacts]]:
        for text in texts:
            yield text, self.process_text(text, language)

    def is_stopword(self, word: str, language: str) -> bool:
        return False

    def is_punct(self, word: str, language: str) -> bool:
        return word in ".,;:!?-()[]{}\"'/\\@#$%^&*"

    def get_supported_entities(self) -> List[str]:
        return []

    def get_supported_languages(self) -> List[str]:
        return ["en"]

## Custom Pattern Recognizers

Two examples of regex-based `PatternRecognizer`:
1. **Employee ID** — matches `EMP-123456`
2. **Project Code** — matches `PRJ-AB-1234` or `PRJ-ABCD-5678`

In [0]:
from presidio_analyzer import Pattern, PatternRecognizer

# ── Example 1: Employee ID ──
employee_id_recognizer = PatternRecognizer(
    supported_entity="EMPLOYEE_ID",
    name="EmployeeIdRecognizer",
    patterns=[
        Pattern(
            name="employee_id_pattern",
            regex=r"\bEMP-\d{6}\b",
            score=0.85,
        )
    ],
    context=["employee", "id", "badge", "staff", "worker"],
    supported_language="en",
)

# ── Example 2: Internal Project Code ──
project_code_recognizer = PatternRecognizer(
    supported_entity="PROJECT_CODE",
    name="ProjectCodeRecognizer",
    patterns=[
        Pattern(
            name="project_code_pattern",
            regex=r"\bFeed[a-z]{2,4}_\d{5}\b",
            score=0.9,
        )
    ],
    context=["project", "code", "initiative", "program"],
    supported_language="en",
)

print("Custom recognizers created:")
print(f"  - EmployeeIdRecognizer: detects EMP-XXXXXX")
print(f"  - ProjectCodeRecognizer: detects Feedxxxx-XXXXX")

Custom recognizers created:
  - EmployeeIdRecognizer: detects EMP-XXXXXX
  - ProjectCodeRecognizer: detects Feedxxxx-XXXXX


## Remote LLM Recognizer (Databricks Foundation Model API)

Detects PERSON and LOCATION entities by calling a Databricks model serving endpoint.
Only active when the `llm_endpoint` widget is set.

In [0]:
from presidio_analyzer import EntityRecognizer


class DatabricksLLMRecognizer(EntityRecognizer):
    """Configuration for LLM-based PII detection.

    Stores endpoint config and supported entities.
    The actual LLM calling logic lives in Cell 22's batched parallel implementation.
    """

    def __init__(self, endpoint_name: str, supported_entities: List[str] = None):
        self.endpoint_name = endpoint_name
        entities = supported_entities or ["PERSON", "LOCATION"]
        super().__init__(
            supported_entities=entities,
            name="DatabricksLLMRecognizer",
            supported_language="en",
        )

    def load(self) -> None:
        pass

    def analyze(self, text, entities, nlp_artifacts=None):
        # No-op: LLM detection is handled by Cell 22's batched parallel implementation
        return []


# Only create if endpoint is configured
llm_recognizer = None
if llm_endpoint:
    llm_recognizer = DatabricksLLMRecognizer(endpoint_name=llm_endpoint)
    print(f"LLM recognizer enabled: endpoint={llm_endpoint}")
else:
    print("LLM recognizer disabled (no endpoint configured)")

LLM recognizer enabled: endpoint=databricks-gpt-5-4-nano


## Build Analyzer Engine

Assembles all recognizers (predefined pattern-based + custom + remote LLM)
into a single `AnalyzerEngine` with the `MinimalNlpEngine`.

In [0]:
from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
from presidio_analyzer.predefined_recognizers import (
    CreditCardRecognizer,
    EmailRecognizer,
    IbanRecognizer,
    IpRecognizer,
    PhoneRecognizer,
    CryptoRecognizer,
)

# Import US recognizers
from presidio_analyzer.predefined_recognizers import (
    UsSsnRecognizer,
    UsPassportRecognizer,
    UsLicenseRecognizer,
    UsBankRecognizer,
    UsItinRecognizer,
)

# Build list of all recognizers
all_recognizers = [
    # -- Generic pattern recognizers --
    CreditCardRecognizer(supported_language="en"),
    EmailRecognizer(supported_language="en"),
    IbanRecognizer(supported_language="en"),
    IpRecognizer(supported_language="en"),
    PhoneRecognizer(supported_language="en"),
    CryptoRecognizer(supported_language="en"),
    # -- US recognizers --
    UsSsnRecognizer(supported_language="en"),
    UsPassportRecognizer(supported_language="en"),
    UsLicenseRecognizer(supported_language="en"),
    UsBankRecognizer(supported_language="en"),
    UsItinRecognizer(supported_language="en"),
    # -- Custom pattern recognizers --
    employee_id_recognizer,
    project_code_recognizer,
]

# Try importing APAC + additional US recognizers (available in 2.2.362)
_optional_recognizers = {
    # US
    "Us": ["UsMbiRecognizer", "UsNpiRecognizer"],
    "Aba": ["AbaRoutingRecognizer"],
    # Singapore
    "Sg": ["SgFinRecognizer", "SgUenRecognizer"],
}

loaded_optional = 0
for _prefix, _classes in _optional_recognizers.items():
    for cls_name in _classes:
        try:
            mod = __import__(
                "presidio_analyzer.predefined_recognizers",
                fromlist=[cls_name],
            )
            cls = getattr(mod, cls_name)
            all_recognizers.append(cls(supported_language="en"))
            loaded_optional += 1
        except (ImportError, AttributeError):
            pass  # Not available in this version

# Add remote LLM recognizer if configured
if llm_recognizer:
    all_recognizers.append(llm_recognizer)

print(f"Recognizers ready:")
print(f"  - {len(all_recognizers)} loaded ({loaded_optional} APAC/optional)")
print(f"  - Supported entities: {sorted(set(e for r in all_recognizers for e in r.supported_entities))}")

Recognizers ready:
  - 19 loaded (5 APAC/optional)
  - Supported entities: ['ABA_ROUTING_NUMBER', 'CREDIT_CARD', 'CRYPTO', 'EMAIL_ADDRESS', 'EMPLOYEE_ID', 'IBAN_CODE', 'IP_ADDRESS', 'LOCATION', 'PERSON', 'PHONE_NUMBER', 'PROJECT_CODE', 'SG_NRIC_FIN', 'SG_UEN', 'US_BANK_NUMBER', 'US_DRIVER_LICENSE', 'US_ITIN', 'US_MBI', 'US_NPI', 'US_PASSPORT', 'US_SSN']


## Context-Aware Enhancement

Boosts detection confidence when a column name contains PII-related keywords.
This compensates for the lack of NLP-based context enhancement (no spaCy).

In [0]:
CONTEXT_BOOST = 0.2  # Score boost when column name matches a PII keyword

COLUMN_CONTEXT_MAP = {
    # column name substring -> entity types to boost
    "email": ["EMAIL_ADDRESS"],
    "mail": ["EMAIL_ADDRESS"],
    "name": ["PERSON"],
    "first_name": ["PERSON"],
    "last_name": ["PERSON"],
    "full_name": ["PERSON"],
    "ssn": ["US_SSN"],
    "social_security": ["US_SSN"],
    "phone": ["PHONE_NUMBER"],
    "mobile": ["PHONE_NUMBER"],
    "cell": ["PHONE_NUMBER"],
    "tel": ["PHONE_NUMBER"],
    "address": ["LOCATION"],
    "city": ["LOCATION"],
    "state": ["LOCATION"],
    "zip": ["LOCATION"],
    "postal": ["LOCATION"],
    "street": ["LOCATION"],
    "passport": ["US_PASSPORT", "IN_PASSPORT", "KR_PASSPORT"],
    "pan": ["IN_PAN"],
    "aadhaar": ["IN_AADHAAR"],
    "aadhar": ["IN_AADHAAR"],
    "gstin": ["IN_GSTIN"],
    "gst": ["IN_GSTIN"],
    "nric": ["SG_NRIC_FIN"],
    "tfn": ["AU_TFN"],
    "medicare": ["AU_MEDICARE"],
    "rrn": ["KR_RRN"],
    "credit_card": ["CREDIT_CARD"],
    "card_number": ["CREDIT_CARD"],
    "cc_num": ["CREDIT_CARD"],
    "iban": ["IBAN_CODE"],
    "ip_addr": ["IP_ADDRESS"],
    "ipv4": ["IP_ADDRESS"],
    "ipv6": ["IP_ADDRESS"],
    "employee_id": ["EMPLOYEE_ID"],
    "emp_id": ["EMPLOYEE_ID"],
    "project_code": ["PROJECT_CODE"],
    "driver_license": ["US_DRIVER_LICENSE", "KR_DRIVER_LICENSE"],
    "license_number": ["US_DRIVER_LICENSE", "KR_DRIVER_LICENSE"],
    "tax_id": ["US_ITIN", "IN_PAN", "AU_TFN"],
    "itin": ["US_ITIN"],
    "routing": ["ABA_ROUTING_NUMBER"],
    "bank_account": ["US_BANK_NUMBER"],
}


def get_context_boost(column_name: str, entity_type: str) -> float:
    """Return score boost if column name matches known PII context."""
    col_lower = column_name.lower()
    for keyword, entity_types in COLUMN_CONTEXT_MAP.items():
        if keyword in col_lower and entity_type in entity_types:
            return CONTEXT_BOOST
    return 0.0


print(f"Context-aware enhancement: +{CONTEXT_BOOST} boost for {len(COLUMN_CONTEXT_MAP)} keyword patterns")

Context-aware enhancement: +0.2 boost for 44 keyword patterns


## Resolve scope to list of tables

In [0]:
def resolve_tables(spark, scope: str) -> list[str]:
    """Parse scope into a list of fully-qualified table names."""
    parts = scope.split(".")
    tables = []

    if len(parts) == 3:
        tables.append(scope)
    elif len(parts) == 2:
        catalog, schema = parts
        rows = spark.sql(f"SHOW TABLES IN `{catalog}`.`{schema}`").collect()
        for row in rows:
            tables.append(f"{catalog}.{schema}.{row.tableName}")
    elif len(parts) == 1:
        catalog = parts[0]
        schemas = spark.sql(f"SHOW SCHEMAS IN `{catalog}`").collect()
        for s in schemas:
            schema_name = s.databaseName
            if schema_name.lower() == "information_schema":
                continue
            try:
                rows = spark.sql(f"SHOW TABLES IN `{catalog}`.`{schema_name}`").collect()
                for row in rows:
                    tables.append(f"{catalog}.{schema_name}.{row.tableName}")
            except Exception as e:
                print(f"  Skipping {catalog}.{schema_name}: {e}")
    else:
        raise ValueError(f"Invalid scope: {scope}. Use catalog, catalog.schema, or catalog.schema.table")

    return tables

tables_to_scan = resolve_tables(spark, scope)

# Exclude the audit results table itself
audit_table_name = f"{scope}.{results_table_override}" if results_table_override else f"{scope}.pii_results"
tables_to_scan = [t for t in tables_to_scan if t != audit_table_name]

print(f"Found {len(tables_to_scan)} table(s) to scan (excluded {audit_table_name}):")
for t in tables_to_scan:
    print(f"  - {t}")

Found 8 table(s) to scan (excluded shao_sandbox1.dbdemos_ai_agent.pii_result):
  - shao_sandbox1.dbdemos_ai_agent.ai_agent_mlflow_eval
  - shao_sandbox1.dbdemos_ai_agent.billing
  - shao_sandbox1.dbdemos_ai_agent.customers
  - shao_sandbox1.dbdemos_ai_agent.dbdemos_ai_agent_demo_payload
  - shao_sandbox1.dbdemos_ai_agent.knowledge_base
  - shao_sandbox1.dbdemos_ai_agent.knowledge_base_raw
  - shao_sandbox1.dbdemos_ai_agent.subscriptions
  - shao_sandbox1.dbdemos_ai_agent.trace_logs_34cf9ec650c94db5b7b95907af9bd8cd


## Read & sample string columns

In [0]:
from pyspark.sql.types import StringType as SparkStringType
import pandas as pd

all_values = []  # (catalog, schema, table, column, value)
table_samples = {}  # table_name -> pandas DataFrame of sampled string columns

for table_name in tables_to_scan:
    parts = table_name.split(".")
    try:
        _df = spark.table(table_name)
    except Exception as e:
        print(f"  Skipping {table_name}: {e}")
        continue

    string_cols = [f.name for f in _df.schema.fields if isinstance(f.dataType, SparkStringType)]
    if not string_cols:
        print(f"  Skipping {table_name}: no string columns")
        continue

    table_samples[table_name] = _df.select(*string_cols).distinct().limit(sample_size).toPandas()

    for _, row in table_samples[table_name].iterrows():
        for c in string_cols:
            val = row[c]
            if pd.notna(val) and val is not None:
                all_values.append((parts[0], parts[1], parts[2], c, str(val)))

    print(f"  {table_name}: {len(table_samples[table_name])} rows, {len(string_cols)} string cols")

print(f"\nTotal values to analyze: {len(all_values)}")

  shao_sandbox1.dbdemos_ai_agent.ai_agent_mlflow_eval: 20 rows, 5 string cols
  shao_sandbox1.dbdemos_ai_agent.billing: 178 rows, 3 string cols
  shao_sandbox1.dbdemos_ai_agent.customers: 500 rows, 11 string cols
  shao_sandbox1.dbdemos_ai_agent.dbdemos_ai_agent_demo_payload: 0 rows, 6 string cols
  shao_sandbox1.dbdemos_ai_agent.knowledge_base: 5 rows, 4 string cols
  shao_sandbox1.dbdemos_ai_agent.knowledge_base_raw: 39 rows, 3 string cols
  shao_sandbox1.dbdemos_ai_agent.subscriptions: 54 rows, 4 string cols
  shao_sandbox1.dbdemos_ai_agent.trace_logs_34cf9ec650c94db5b7b95907af9bd8cd: 42 rows, 7 string cols

Total values to analyze: 6755


## Run PII detection (pattern + LLM)


In [0]:
from collections import defaultdict
import time, json, re
from concurrent.futures import ThreadPoolExecutor, as_completed

# Each detection is a dict: {score, source, recognizer, matched_text}
detections = defaultdict(list)
total = len(all_values)

# ===================================================================
# Phase 1: Pattern-only analysis (fast -- no LLM overhead)
# ===================================================================
_pattern_recs = [r for r in all_recognizers if not isinstance(r, DatabricksLLMRecognizer)]
_pattern_analyzer = AnalyzerEngine(
    registry=RecognizerRegistry(recognizers=_pattern_recs, supported_languages=["en"]),
    nlp_engine=MinimalNlpEngine(),
    supported_languages=["en"],
)

_llm_only_entities = {"PERSON", "LOCATION"}
_pattern_entities = [e for e in ALLOWED_ENTITY_TYPES if e not in _llm_only_entities]

# Value-level filters: skip values that are clearly non-PII
_DATE_RE = re.compile(r"^\d{4}-\d{2}-\d{2}")
_NUMERIC_RE = re.compile(r"^-?\d+\.?\d*$")

t0 = time.time()
for catalog, schema, table, column, value in all_values:
    text = f"{table} {column}: {value}"
    results = _pattern_analyzer.analyze(text=text, entities=_pattern_entities, language="en")
    seen = {}
    for r in results:
        if r.entity_type not in seen or r.score > seen[r.entity_type][0]:
            rec_name = (r.recognition_metadata or {}).get("recognizer_name", "unknown")
            matched = text[r.start:r.end]
            seen[r.entity_type] = (r.score, rec_name, matched)
    for entity_type, (score, rec_name, matched) in seen.items():
        boost = get_context_boost(column, entity_type)
        detections[(catalog, schema, table, column, entity_type)].append({
            "score": min(1.0, score + boost),
            "source": "pattern",
            "recognizer": rec_name,
            "matched_text": matched,
        })

t_pattern = time.time() - t0
print(f"Phase 1 (patterns): {total} values in {t_pattern:.1f}s ({total / max(t_pattern, 0.001):.0f} values/s)")
print(f"  Found {len(detections)} (column, entity) pairs")

# ===================================================================
# Phase 2: Batched + parallel LLM detection
# ===================================================================
LLM_WORKERS = 8
LLM_SAMPLE_PER_COL = 50

t_llm = 0.0
if llm_recognizer:
    llm_entities = set(llm_recognizer.supported_entities)

    # Skip only columns that already passed the threshold in pattern analysis
    detected_cols = set()
    for (c, s, t, col, _etype), dets in detections.items():
        if any(d["score"] > float(score_threshold) for d in dets):
            detected_cols.add((c, s, t, col))

    # Collect unique values per column, excluding passed-threshold, numeric, and dates
    col_unique = defaultdict(set)
    skipped_values = 0
    for cat, sch, tbl, col, val in all_values:
        if (cat, sch, tbl, col) in detected_cols:
            continue
        if _NUMERIC_RE.match(val) or _DATE_RE.match(val):
            skipped_values += 1
            continue
        col_unique[(cat, sch, tbl, col)].add(val)

    col_unique = {k: v for k, v in col_unique.items() if v}

    print(f"\nPhase 2 pre-filter: {len(detected_cols)} columns already passed threshold in patterns, {skipped_values} numeric/date values skipped")

    # One batch per column -- never mix columns in a single LLM call
    batches = [
        [(*key, v) for v in list(vals)[:LLM_SAMPLE_PER_COL]]
        for key, vals in col_unique.items()
    ]

    llm_value_count = sum(len(b) for b in batches)

    if batches:
        print(f"Phase 2 (LLM): {llm_value_count} values -> {len(batches)} batches (1 column each, {LLM_WORKERS} threads)")

        from databricks.sdk import WorkspaceClient
        from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
        import threading
        _w = WorkspaceClient()
        _print_lock = threading.Lock()

        def _llm_batch(batch_idx, batch):
            col_label = f"{batch[0][2]}.{batch[0][3]}"  # table.column
            lines = "\n".join(
                f"{i+1}. {val}"
                for i, (_, _, tbl, col, val) in enumerate(batch)
            )
            prompt = f"""Extract person names and addresses from each numbered text (all from column [{col_label}]).

IMPORTANT: Use the column name as a contextual signal.
Return ONLY a JSON array. Each object must have:
- "index": line number (1-based)
- "entity_type": "PERSON" or "ADDRESS"
- "text": exact matched substring
- "score": confidence float 0.0-1.0 where:
    0.9-1.0 = clearly PII (e.g. "John Smith", "123 Main St, New York") or column name strongly implies PII
    0.7-0.89 = likely PII but ambiguous (e.g. single common first name, country name in a generic column)
    0.5-0.69 = possibly PII, uncertain
    below 0.5 = unlikely PII

Do NOT flag database paths, table names, schema names, UUIDs, or technical identifiers.
Return [] if ABSOLUTELY no PII found, return records even for low confidence values.

{lines}

JSON:"""
            raw_response = ""
            parsed = []
            try:
                resp = _w.serving_endpoints.query(
                    name=llm_recognizer.endpoint_name,
                    messages=[
                        ChatMessage(role=ChatMessageRole.USER, content=prompt),
                    ],
                    max_tokens=1024,
                    temperature=0.0,
                )
                raw_response = resp.choices[0].message.content.strip()
                m = re.search(r"\[.*\]", raw_response, re.DOTALL)
                parsed = json.loads(m.group()) if m else []
            except Exception as e:
                raw_response = f"ERROR: {e}"

            if llm_debug:
                with _print_lock:
                    print(f"\n{'='*60}")
                    print(f"LLM Batch {batch_idx + 1}/{len(batches)} -- [{col_label}] ({len(batch)} values)")
                    print(f"{'='*60}")
                    print(f"INPUT:\n{lines}")
                    print(f"\nOUTPUT:\n{raw_response}")
                    print(f"\nPARSED: {len(parsed)} detections")
                    for d in parsed:
                        print(f"  #{d.get('index')} {d.get('entity_type')}: \"{d.get('text')}\" (score: {d.get('score')})")
                    print(f"{'='*60}")

            return parsed

        t_llm_start = time.time()
        completed = 0

        with ThreadPoolExecutor(max_workers=LLM_WORKERS) as executor:
            future_map = {executor.submit(_llm_batch, idx, batch): batch for idx, batch in enumerate(batches)}
            for future in as_completed(future_map):
                batch = future_map[future]
                completed += 1
                try:
                    for det in future.result():
                        idx = det.get("index", 0) - 1
                        if 0 <= idx < len(batch):
                            cat, sch, tbl, col, _ = batch[idx]
                            etype = det.get("entity_type", "")
                            matched = det.get("text", "")
                            raw_score = det.get("score")
                            if raw_score is None:
                                continue
                            score = float(raw_score)
                            if etype in llm_entities:
                                boost = get_context_boost(col, etype)
                                detections[(cat, sch, tbl, col, etype)].append({
                                    "score": min(1.0, score + boost),
                                    "source": "llm",
                                    "recognizer": "DatabricksLLMRecognizer",
                                    "matched_text": matched,
                                })
                except Exception:
                    pass

        t_llm = time.time() - t_llm_start
    else:
        print(f"Phase 2 (LLM): skipped -- no columns need LLM analysis")

print(f"\nDone: {len(detections)} (column, entity) pairs")
print(f"  Pattern: {t_pattern:.1f}s | LLM: {t_llm:.1f}s | Total: {t_pattern + t_llm:.1f}s")

Phase 1 (patterns): 6755 values in 4.7s (1432 values/s)
  Found 24 (column, entity) pairs

Phase 2 pre-filter: 11 columns already passed threshold in patterns, 500 numeric/date values skipped
Phase 2 (LLM): 473 values -> 25 batches (1 column each, 8 threads)

Done: 24 (column, entity) pairs
  Pattern: 4.7s | LLM: 6.8s | Total: 11.5s


## Structured Engine Analysis

Uses `presidio_structured` to analyze DataFrames column-by-column as an alternative view.
Results are merged with pattern-based detections.

In [0]:
try:
    from presidio_structured import PandasAnalysisBuilder
    import logging

    # Suppress warnings for PERSON/LOCATION (no pattern recognizer; handled by LLM in Phase 2)
    logging.getLogger("presidio-analyzer").setLevel(logging.ERROR)

    def _structured_analyze(table_name, pdf):
        """Analyze a single table's sample DataFrame (runs in thread)."""
        local_builder = PandasAnalysisBuilder(analyzer=_pattern_analyzer)
        analysis = local_builder.generate_analysis(pdf, language="en")
        return table_name, analysis.entity_mapping

    t0 = time.time()
    results = []
    n_workers = min(8, len(table_samples))

    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = {
            executor.submit(_structured_analyze, name, pdf): name
            for name, pdf in table_samples.items()
        }
        for future in as_completed(futures):
            name = futures[future]
            try:
                results.append(future.result())
            except Exception as e:
                print(f"  {name}: structured analysis failed -- {e}")

    # Cross-validate: structured engine confirms or supplements Cell 22 detections
    print("Structured engine analysis per table:")
    confirmed, added = 0, 0
    for table_name, entity_mapping in sorted(results, key=lambda x: x[0]):
        parts = table_name.split(".")
        print(f"\n  {table_name}:")
        for col_name, entity_type in entity_mapping.items():
            if entity_type not in ALLOWED_ENTITY_TYPES:
                continue
            key = (parts[0], parts[1], parts[2], col_name, entity_type)
            if key in detections:
                # Cross-validation: structured engine confirms Cell 22's detection
                existing_scores = [d["score"] for d in detections[key]]
                avg = sum(existing_scores) / len(existing_scores)
                print(f"    {col_name} -> {entity_type} (confirmed, avg score: {avg:.2f})")
                confirmed += 1
            else:
                # Structured engine found something Cell 22 missed
                # Use context boost as the only signal since no pattern/LLM score exists
                boost = get_context_boost(col_name, entity_type)
                if boost > 0:
                    detections[key].append({
                        "score": boost,
                        "source": "structured",
                        "recognizer": "PandasAnalysisBuilder",
                        "matched_text": "",
                    })
                    print(f"    {col_name} -> {entity_type} (new, context-only score: {boost})")
                    added += 1
                else:
                    print(f"    {col_name} -> {entity_type} (no pattern/LLM/context score, skipping)")

    logging.getLogger("presidio-analyzer").setLevel(logging.WARNING)

    elapsed = time.time() - t0
    print(f"\nStructured analysis: {len(table_samples)} tables in {elapsed:.1f}s ({n_workers} threads)")
    print(f"  Confirmed: {confirmed}, New (context-only): {added}")
    print(f"  Total: {len(detections)} (column, entity) pairs")

except ImportError:
    print("presidio_structured not available -- skipping structured engine analysis")

# ===================================================================
# Context-based column name scoring for columns without passing detections
# ===================================================================
# For every scanned column that has NO detection (or only below-threshold
# detections), check if the column name matches COLUMN_CONTEXT_MAP.
# If so, record a context-only entry (score = CONTEXT_BOOST).
# This ensures PII-suggestive column names always appear in the audit table.

all_columns = set()
for cat, sch, tbl, col, _ in all_values:
    all_columns.add((cat, sch, tbl, col))

context_added = 0
for cat, sch, tbl, col in all_columns:
    col_lower = col.lower()
    for keyword, entity_types in COLUMN_CONTEXT_MAP.items():
        if keyword in col_lower:
            for entity_type in entity_types:
                key = (cat, sch, tbl, col, entity_type)
                # Only add if no existing detection for this (column, entity) pair
                if key not in detections:
                    detections[key].append({
                        "score": CONTEXT_BOOST,
                        "source": "context",
                        "recognizer": "ColumnNameMatcher",
                        "matched_text": "",
                    })
                    context_added += 1

print(f"\nContext-based column name scoring: {context_added} potential PII candidate(s) added from column name matching")
print(f"  Final total: {len(detections)} (column, entity) pairs")

Structured engine analysis per table:

  shao_sandbox1.dbdemos_ai_agent.ai_agent_mlflow_eval:
    dataset_record_id -> US_DRIVER_LICENSE (confirmed, avg score: 0.30)
    created_by -> EMAIL_ADDRESS (confirmed, avg score: 1.00)
    last_updated_by -> EMAIL_ADDRESS (confirmed, avg score: 1.00)
    inputs -> EMAIL_ADDRESS (confirmed, avg score: 1.00)
    expectations -> EMAIL_ADDRESS (confirmed, avg score: 1.00)

  shao_sandbox1.dbdemos_ai_agent.billing:

  shao_sandbox1.dbdemos_ai_agent.customers:
    email -> EMAIL_ADDRESS (confirmed, avg score: 1.00)
    phone -> PHONE_NUMBER (confirmed, avg score: 0.60)
    address -> PHONE_NUMBER (confirmed, avg score: 0.40)

  shao_sandbox1.dbdemos_ai_agent.dbdemos_ai_agent_demo_payload:

  shao_sandbox1.dbdemos_ai_agent.knowledge_base:
    content -> US_DRIVER_LICENSE (confirmed, avg score: 0.30)

  shao_sandbox1.dbdemos_ai_agent.knowledge_base_raw:

  shao_sandbox1.dbdemos_ai_agent.subscriptions:

  shao_sandbox1.dbdemos_ai_agent.trace_logs_34cf9e

## Aggregate & filter

In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Build summary rows from enriched detections -- go straight to Spark DataFrame
summary_rows = []
for (catalog, schema, table, column, entity_type), dets in detections.items():
    scores = [d["score"] for d in dets]
    sources = sorted(set(d["source"] for d in dets))
    recognizers = sorted(set(d["recognizer"] for d in dets))
    sample_matches = list(set(d["matched_text"] for d in dets if d["matched_text"]))[:3]

    total = len(scores)
    avg_score = sum(scores) / total if total > 0 else 0.0

    reason_parts = []
    if "pattern" in sources:
        pat_recs = sorted(set(d["recognizer"] for d in dets if d["source"] == "pattern"))
        reason_parts.append(f"Pattern match by {', '.join(pat_recs)}")
    if "llm" in sources:
        reason_parts.append(f"LLM detected via {llm_endpoint}")
    if "structured" in sources:
        reason_parts.append("Structured column-level analysis")

    # Explain context boost if applied
    boost = get_context_boost(column, entity_type)
    if boost > 0:
        matched_kw = [kw for kw in COLUMN_CONTEXT_MAP if kw in column.lower() and entity_type in COLUMN_CONTEXT_MAP[kw]]
        kw_str = f"'{matched_kw[0]}'" if matched_kw else "PII keyword"
        reason_parts.append(f"Context boost +{boost} (column name contains {kw_str})")

    if sample_matches:
        reason_parts.append(f"Examples: {'; '.join(sample_matches)}")

    summary_rows.append({
        "table_catalog": catalog,
        "table_schema": schema,
        "table_name": table,
        "column_name": column,
        "entity_type": entity_type,
        "tag_value": ENTITY_TO_TAG.get(entity_type) or "none",
        "score": float(avg_score),
        "max_score": float(max(scores)),
        "sources": ", ".join(sources),
        "recognizers": ", ".join(recognizers),
        "sample_matches": "; ".join(sample_matches) if sample_matches else "",
        "reason": ". ".join(reason_parts),
        "detection_count": total,
        "full_table_name": f"{catalog}.{schema}.{table}",
        "identification_date": datetime.now(),
    })

audit_df = None
if not summary_rows:
    print("No PII detected -- nothing to save")
else:
    all_detections_df = spark.createDataFrame(summary_rows)

    # Rank: keep top entity per column
    w = Window.partitionBy("full_table_name", "column_name").orderBy(F.desc("score"))
    ranked_df = (
        all_detections_df
        .withColumn("_rank", F.row_number().over(w))
        .filter(F.col("_rank") == 1)
        .drop("_rank")
    )

    # Add passed_threshold flag -- all rows are saved, only passed ones get tagged
    audit_df = ranked_df.withColumn(
        "passed_threshold",
        F.col("score") > float(score_threshold)
    )

    passed_df = audit_df.filter(F.col("passed_threshold") == True)
    low_conf_df = audit_df.filter(F.col("passed_threshold") == False)
    passed_count = passed_df.count()
    low_conf_count = low_conf_df.count()

    # Display passed detections
    print(f"PII detected in {passed_count} column(s) (score > {score_threshold}, will be tagged):")
    if passed_count > 0:
        display(passed_df.select(
            "full_table_name", "column_name", "entity_type", "tag_value",
            "score", "sources", "reason", "passed_threshold"
        ))

    # Display low-confidence detections
    if low_conf_count > 0:
        print(f"\n--- Low-confidence detections ({low_conf_count} column(s), score <= {score_threshold}, will NOT be tagged) ---")
        display(low_conf_df.select(
            "full_table_name", "column_name", "entity_type", "tag_value",
            "score", "sources", "reason", "passed_threshold"
        ).orderBy(F.desc("score")))
    else:
        print(f"\nNo low-confidence detections below threshold ({score_threshold}).")

    # Save ALL detections (passed + low-confidence) to audit table
    audit_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(audit_table_name)
    print(f"\nAll {passed_count + low_conf_count} detection(s) saved to {audit_table_name} (with passed_threshold flag)")

PII detected in 11 column(s) (score > 0.9, will be tagged):


full_table_name,column_name,entity_type,tag_value,score,sources,reason,passed_threshold
shao_sandbox1.dbdemos_ai_agent.ai_agent_mlflow_eval,created_by,EMAIL_ADDRESS,email,1.0,pattern,Pattern match by EmailRecognizer. Examples: steve.shao@databricks.com,true
shao_sandbox1.dbdemos_ai_agent.ai_agent_mlflow_eval,expectations,EMAIL_ADDRESS,email,1.0,pattern,Pattern match by EmailRecognizer. Examples: tamirodriguez@example.org; gibsonolivia@example.net; smitchell@example.net,true
shao_sandbox1.dbdemos_ai_agent.ai_agent_mlflow_eval,inputs,EMAIL_ADDRESS,email,1.0,pattern,Pattern match by EmailRecognizer. Examples: tamirodriguez@example.org; gibsonolivia@example.net; smitchell@example.net,true
shao_sandbox1.dbdemos_ai_agent.ai_agent_mlflow_eval,last_updated_by,EMAIL_ADDRESS,email,1.0,pattern,Pattern match by EmailRecognizer. Examples: steve.shao@databricks.com,true
shao_sandbox1.dbdemos_ai_agent.customers,email,EMAIL_ADDRESS,email,1.0,pattern,Pattern match by EmailRecognizer. Context boost +0.2 (column name contains 'email'). Examples: kellychristopher@example.org; vickiwood@example.org; ymitchell@example.com,true
shao_sandbox1.dbdemos_ai_agent.customers,phone,US_NPI,none,1.0,pattern,Pattern match by UsNpiRecognizer. Examples: 2438315410,true
shao_sandbox1.dbdemos_ai_agent.knowledge_base,content,EMAIL_ADDRESS,email,1.0,pattern,Pattern match by EmailRecognizer. Examples: support@smarthome.com; support@telco.com; support@company.com,true
shao_sandbox1.dbdemos_ai_agent.trace_logs_34cf9ec650c94db5b7b95907af9bd8cd,request,EMAIL_ADDRESS,email,1.0,pattern,Pattern match by EmailRecognizer. Examples: tamirodriguez@example.org; gibsonolivia@example.net; smitchell@example.net,true
shao_sandbox1.dbdemos_ai_agent.trace_logs_34cf9ec650c94db5b7b95907af9bd8cd,request_preview,EMAIL_ADDRESS,email,1.0,pattern,Pattern match by EmailRecognizer. Examples: tamirodriguez@example.org; gibsonolivia@example.net; smitchell@example.net,true
shao_sandbox1.dbdemos_ai_agent.trace_logs_34cf9ec650c94db5b7b95907af9bd8cd,response,EMAIL_ADDRESS,email,1.0,pattern,Pattern match by EmailRecognizer. Examples: tamirodriguez@example.org; gibsonolivia@example.net; smitchell@example.net,true



--- Low-confidence detections (11 column(s), score <= 0.9, will NOT be tagged) ---


full_table_name,column_name,entity_type,tag_value,score,sources,reason,passed_threshold
shao_sandbox1.dbdemos_ai_agent.customers,address,PHONE_NUMBER,phone,0.4000000000000001,pattern,Pattern match by PhoneRecognizer. Examples: 1687; 1686; 16475,false
shao_sandbox1.dbdemos_ai_agent.ai_agent_mlflow_eval,dataset_record_id,US_DRIVER_LICENSE,none,0.3,pattern,Pattern match by UsLicenseRecognizer. Examples: cb434938; b786; a068,false
shao_sandbox1.dbdemos_ai_agent.customers,city,LOCATION,address,0.2,context,Context boost +0.2 (column name contains 'city'),false
shao_sandbox1.dbdemos_ai_agent.customers,first_name,PERSON,name,0.2,context,Context boost +0.2 (column name contains 'name'),false
shao_sandbox1.dbdemos_ai_agent.customers,last_name,PERSON,name,0.2,context,Context boost +0.2 (column name contains 'name'),false
shao_sandbox1.dbdemos_ai_agent.customers,state,LOCATION,address,0.2,context,Context boost +0.2 (column name contains 'state'),false
shao_sandbox1.dbdemos_ai_agent.customers,zip_code,LOCATION,address,0.2,context,Context boost +0.2 (column name contains 'zip'),false
shao_sandbox1.dbdemos_ai_agent.knowledge_base,product_name,PERSON,name,0.2,context,Context boost +0.2 (column name contains 'name'),false
shao_sandbox1.dbdemos_ai_agent.knowledge_base_raw,file_name,PERSON,name,0.2,context,Context boost +0.2 (column name contains 'name'),false
shao_sandbox1.dbdemos_ai_agent.subscriptions,plan_name,PERSON,name,0.2,context,Context boost +0.2 (column name contains 'name'),false



All 22 detection(s) saved to shao_sandbox1.dbdemos_ai_agent.pii_result (with passed_threshold flag)


## Apply UC tags

In [0]:
if audit_df is not None:
    # Only tag columns that passed the score threshold
    passed_df = audit_df.filter(F.col("passed_threshold") == True)
    tagged = 0
    for r in passed_df.collect():
        table = r["full_table_name"]
        column = r["column_name"]
        entity = r["entity_type"]
        tag_value = r["tag_value"]
        if tag_value is None or tag_value == "none":
            print(f"  Skipping {table}.{column} ({entity}) -- no tag mapping")
            continue
        print(f"  Tagging {table}.`{column}` -> pii='{tag_value}'")
        spark.sql(f"ALTER TABLE {table} ALTER COLUMN `{column}` SET TAGS ('pii' = '{tag_value}')")
        tagged += 1

    print(f"\nApplied PII tags to {tagged} column(s) (passed threshold)")
else:
    print("No PII detected -- nothing to tag")

  Tagging shao_sandbox1.dbdemos_ai_agent.ai_agent_mlflow_eval.`created_by` -> pii='email'
  Tagging shao_sandbox1.dbdemos_ai_agent.ai_agent_mlflow_eval.`expectations` -> pii='email'
  Tagging shao_sandbox1.dbdemos_ai_agent.ai_agent_mlflow_eval.`inputs` -> pii='email'
  Tagging shao_sandbox1.dbdemos_ai_agent.ai_agent_mlflow_eval.`last_updated_by` -> pii='email'
  Tagging shao_sandbox1.dbdemos_ai_agent.customers.`email` -> pii='email'
  Skipping shao_sandbox1.dbdemos_ai_agent.customers.phone (US_NPI) -- no tag mapping
  Tagging shao_sandbox1.dbdemos_ai_agent.knowledge_base.`content` -> pii='email'
  Tagging shao_sandbox1.dbdemos_ai_agent.trace_logs_34cf9ec650c94db5b7b95907af9bd8cd.`request` -> pii='email'
  Tagging shao_sandbox1.dbdemos_ai_agent.trace_logs_34cf9ec650c94db5b7b95907af9bd8cd.`request_preview` -> pii='email'
  Tagging shao_sandbox1.dbdemos_ai_agent.trace_logs_34cf9ec650c94db5b7b95907af9bd8cd.`response` -> pii='email'
  Tagging shao_sandbox1.dbdemos_ai_agent.trace_logs_34cf9e